# 05 — Script Generation

Turn one saved `VideoOutline` into complete, structured narration for an
educational short video.

This notebook loads an outline, generates narration, normalizes timing from
actual word count, saves the validated script, and previews the result.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.outlines import load_outline
from educational_shorts.prompts import load_prompt
from educational_shorts.scripts import (
    build_script_filename,
    find_outline_file,
    generate_script,
    save_script,
)

print(f"Project root: {PROJECT_ROOT}")

## Configuration

In [ ]:
OUTLINES_DIRECTORY = PROJECT_ROOT / "data" / "outlines"
SCRIPTS_DIRECTORY = PROJECT_ROOT / "data" / "scripts"

# Set a specific filename, or leave as None to use the newest outline.
OUTLINE_FILENAME = None

TARGET_WORDS_PER_MINUTE = 145
TEMPERATURE = 0.5
GENERATION_SEED = 42

print(f"Outlines directory: {OUTLINES_DIRECTORY}")
print(f"Scripts directory: {SCRIPTS_DIRECTORY}")

## Load an outline

In [ ]:
outline_path = find_outline_file(
    outlines_directory=OUTLINES_DIRECTORY,
    filename=OUTLINE_FILENAME,
)

video_outline = load_outline(outline_path)

print(f"Loaded outline from: {outline_path}")
print(f"Topic: {video_outline.topic.title}")
print(
    f"Outline target: {video_outline.estimated_total_seconds} seconds "
    f"across {len(video_outline.sections)} body sections"
)

## Load the script-generation prompt

In [ ]:
script_system_prompt = load_prompt("script_generation")
print("Script-generation prompt loaded.")

## Generate the script

In [ ]:
video_script = generate_script(
    outline=video_outline,
    system_prompt=script_system_prompt,
    target_wpm=TARGET_WORDS_PER_MINUTE,
    temperature=TEMPERATURE,
    seed=GENERATION_SEED,
)

print(f"Generated {video_script.word_count} words.")
print(
    f"Estimated spoken duration: "
    f"{video_script.estimated_total_seconds} seconds"
)

## Save the script

In [ ]:
output_path = SCRIPTS_DIRECTORY / build_script_filename(video_outline)

save_script(video_script, output_path)
print(f"Saved script to {output_path}")

## Preview structured segments

In [ ]:
print(f"TITLE: {video_script.topic.title}")
print()

print(f"HOOK ({video_script.hook.estimated_seconds}s)")
print(video_script.hook.narration)
print(f"Visual: {video_script.hook.visual_direction}")
print()

for index, section in enumerate(video_script.sections, start=1):
    print(
        f"{index}. {section.segment_type.upper()} "
        f"({section.estimated_seconds}s)"
    )
    print(section.narration)
    print(f"Visual: {section.visual_direction}")
    print()

print(f"CLOSING ({video_script.closing.estimated_seconds}s)")
print(video_script.closing.narration)
print(f"Visual: {video_script.closing.visual_direction}")
print()

print(f"WORD COUNT: {video_script.word_count}")
print(f"ESTIMATED TOTAL: {video_script.estimated_total_seconds} seconds")

## Preview complete narration

In [ ]:
print(video_script.full_narration)